In [ ]:
%pip install langgraph langchain langchain-mcp-adapters langchain-openai "pydantic[email]" mcp fastmcp "fastmcp[tasks]"

In [1]:
import langgraph
import importlib.metadata

version = importlib.metadata.version("langgraph")
print(f"LangGraph Version: {version}")

LangGraph Version: 1.1.6


##### Work With State Graphs
LangGraph is all about implementing LLM applications as directed graphs. You can think of a directed graph as a sequence of instructions composed of nodes and edges, that tell you how to complete a task. 

In LangGraph, 
- `nodes` represent actions that your graph can take, such as calling a function, and
- `edges` tell you which node to go to next.

To understand this better, take a look at this directed graph:

![](./notice_extraction_graph.png)

##### LangChain Chains and Their Limitations
Suppose you work for a large real estate development company. Your company receives hundreds of emails a day from regulatory entities and other organizations regarding active construction sites. For instance, your company might receive a notice from an inspector saying that a construction site doesn’t comply with safety regulations.

Your job is to build a tool that can read these emails, extract critical information from them, and notify the correct internal team who will take action. The first step you’ll take to accomplish this is to build a LangChain chain that uses an LLM to extract structured fields from a regulatory notice email. You start by defining a Pydantic BaseModel that describes all the fields you want to extract from the email:



In [ ]:
from com.example.rag.graphs.notice_extraction import NOTICE_EXTRACTION_GRAPH
from IPython.display import Image, display

image_data = NOTICE_EXTRACTION_GRAPH.get_graph().draw_mermaid_png()
# with open("notice_extraction_graph.png", mode="wb") as f:
#     f.write(image_data)

#
try:
  display(Image(image_data))
except Exception:
  pass


In [ ]:
from com.example.rag.graphs.notice_extraction import NOTICE_EXTRACTION_GRAPH, NoticeEmailExtract
from example_emails import EMAILS

initial_state = {
     "notice_message": EMAILS[0],
     "notice_email_extract": None,
     "escalation_text_criteria": """There's a risk of fire or water damage at the site""",
     "escalation_dollar_criteria": 100_000,
     "requires_escalation": False,
     "escalation_emails": ["brog@abc.com", "bigceo@company.com"],
}

final_state = NOTICE_EXTRACTION_GRAPH.invoke(initial_state)




In [ ]:

final_state["notice_email_extract"]

NoticeEmailExtract(
    date_of_notice=datetime.date(2024, 10, 15),
    entity_name='Occupational Safety and Health Administration (OSHA)',
    entity_phone='(555) 123-4567',
    entity_email='compliance.osha@osha.gov',
    project_id=111232345,
    site_location='123 Main Street, Dallas, TX',
    violation_type='Lack of fall protection, Unsafe scaffolding setup, Inadequate personal protective equipment (PPE)',
    required_changes='Install guardrails and fall arrest systems on all scaffolding over 10 feet. Conduct an inspection of all \
        scaffolding structures and reinforce unstable sections. Ensure all workers on-site are provided with necessary PPE and conduct \
        safety training on proper usage.',
    compliance_deadline=datetime.date(2024, 11, 10),
    max_potential_fine=25000.0
)

final_state["requires_escalation"]

##### Test Your Graph Agent

>>> from graphs.email_agent import email_agent_graph
>>> from example_emails import EMAILS

>>> message_1 = {"messages": [("human", EMAILS[1])]}

>>> for chunk in email_agent_graph.stream(message_1, stream_mode="values"):
...      chunk["messages"][-1].pretty_print()
...
================================ Human Message ============================


    From: debby@stack.com

    Hey Betsy,

    Here's your invoice for $1000 for the cookies you ordered.

================================== Ai Message =============================
Tool Calls:
  determine_email_action (call_MWiiVRHsEHKMX6lftRwJEZyO)
 Call ID: call_MWiiVRHsEHKMX6lftRwJEZyO
  Args:
    email: From: debby@stack.com

Hey Betsy,

Here's your invoice for $1000 for the cookies you ordered.
================================= Tool Message ============================
Name: determine_email_action


    If the email appears to be an invoice of any kind or related to
    billing, forward the email to the billing and invoices team:
    billing@company.com and send a wrong email notice back to the sender.
    The correct department is
    billing@company.com.

    If the email appears to be from a customer, forward to
    support@company.com,
    cdetuma@company.com, and ctu@abc.com. Send a wrong email
    notice back to the customer and let them know the correct
    department is support@company.com.

    For any other emails, please send a wrong email notification
    and try to infer the correct department from one of
    billing@company.com, support@company.com,
    humanresources@company.com, and it@company.com.

================================== Ai Message =============================
Tool Calls:
  forward_email (call_PTXylQoEEdKfLS680bPCd9WO)
 Call ID: call_PTXylQoEEdKfLS680bPCd9WO
  Args:
    email_message: From: debby@stack.com

Hey Betsy,

Here's your invoice for $1000 for the cookies you ordered.
    send_to_email: billing@company.com
  send_wrong_email_notification_to_sender (call_NGpIerzPsToKSK3u2JZo5RwW)
 Call ID: call_NGpIerzPsToKSK3u2JZo5RwW
  Args:
    sender_email: debby@stack.com
    correct_department: billing@company.com
2025-01-26 12:00:46,317 - INFO - Forwarding the email to
billing@company.com...
2025-01-26 12:00:46,318 - INFO - Sending wrong email notification
to debby@stack.com...
2025-01-26 12:00:48,323 - INFO - Email forwarded!
2025-01-26 12:00:48,324 - INFO - Email sent!
================================= Tool Message ============================
Name: send_wrong_email_notification_to_sender

true
================================== Ai Message =============================

The email has been successfully forwarded to the billing and invoices team,
and a notification
has been sent to the sender, informing them that the correct department is
billing@company.com.

>>> message_2 = {"messages": [("human", EMAILS[2])]}

>>> for chunk in email_agent_graph.stream(message_2, stream_mode="values"):
...      chunk["messages"][-1].pretty_print()
...
================================ Human Message ===========================


    From: tdavid@companyxyz.com

    Hi Paul,

    We have an issue with the HVAC system your team installed in
    apartment 1235. We'd like to request maintenance or a refund.

    Thanks,
    Terrance

================================== Ai Message ============================
Tool Calls:
  determine_email_action (call_QQwWoaqG3dKc6AjJ92mc6iXI)
 Call ID: call_QQwWoaqG3dKc6AjJ92mc6iXI
  Args:
    email: From: tdavid@companyxyz.com

Hi Paul,

We have an issue with the HVAC system your team installed in
apartment 1235. We'd like to request maintenance or a refund.

Thanks,
Terrance
================================= Tool Message ===========================
Name: determine_email_action


    If the email appears to be an invoice of any kind or related to
    billing, forward the email to the billing and invoices team:
    billing@company.com
    and send a wrong email notice back to the sender. The correct
    department is
    billing@company.com.

    If the email appears to be from a customer, forward to
    support@company.com,
    cdetuma@company.com, and ctu@abc.com. Be sure to forward it to
    all three emails listed.
    Send a wrong email notice back to the
    customer and let them know the correct department is
    support@company.com.

    For any other emails, please send a wrong email notification
    and try to
    infer the correct department from one of billing@company.com,
    support@company.com,
    humanresources@company.com, and it@company.com.

================================== Ai Message ============================
Tool Calls:
  forward_email (call_vvlAiZrF6CyNlD6pOOSzfCIr)
 Call ID: call_vvlAiZrF6CyNlD6pOOSzfCIr
  Args:
    email_message: From: tdavid@companyxyz.com

Hi Paul,

We have an issue with the HVAC system your team installed in
apartment 1235. We'd like to request maintenance or a refund.

Thanks,
Terrance
    send_to_email: support@company.com
  forward_email (call_Pk9b89g1gUty9mMqvo9luZSD)
 Call ID: call_Pk9b89g1gUty9mMqvo9luZSD
  Args:
    email_message: From: tdavid@companyxyz.com

Hi Paul,

We have an issue with the HVAC system your team installed in
apartment 1235. We'd like to request maintenance or a refund.

Thanks,
Terrance
    send_to_email: cdetuma@company.com
  forward_email (call_4JLEOKpb8qOHgdmbDohH071p)
 Call ID: call_4JLEOKpb8qOHgdmbDohH071p
  Args:
    email_message: From: tdavid@companyxyz.com

Hi Paul,

We have an issue with the HVAC system your team installed in
apartment 1235. We'd like to request maintenance or a refund.

Thanks,
Terrance
    send_to_email: ctu@abc.com
  send_wrong_email_notification_to_sender (call_tRimFJO4ElR8g8P96Eqi8EPC)
 Call ID: call_tRimFJO4ElR8g8P96Eqi8EPC
  Args:
    sender_email: tdavid@companyxyz.com
    correct_department: support@company.com
2025-01-26 12:10:21,796 - INFO - Forwarding the email to
support@company.com...
2025-01-26 12:10:21,797 - INFO - Forwarding the email to
cdetuma@company.com...
2025-01-26 12:10:21,797 - INFO - Sending wrong email notification
to tdavid@companyxyz.com...
2025-01-26 12:10:21,798 - INFO - Forwarding the email to ctu@abc.com...
2025-01-26 12:10:23,801 - INFO - Email sent!
2025-01-26 12:10:23,802 - INFO - Email forwarded!
2025-01-26 12:10:23,802 - INFO - Email forwarded!
2025-01-26 12:10:23,804 - INFO - Email forwarded!
================================= Tool Message ===========================
Name: send_wrong_email_notification_to_sender

true
================================== Ai Message ===========================

The email from Terrance has been successfully forwarded to the support team,
as well as to the
additional contacts. A notification has also been sent to Terrance informing
him that the correct
the department for his request is support@company.com.

>>> escalation_criteria = """"There's an immediate risk of electrical,
... water, or fire damage"""

>>> message_with_criteria = f"""
... The escalation criteria is: {escalation_criteria}
...
... Here's the email:
...
... {EMAILS[3]}
... """
>>> message_3 = {"messages": [("human", message_with_criteria)]}

>>> for chunk in email_agent_graph.stream(message_3, stream_mode="values"):
...     chunk["messages"][-1].pretty_print()
...
================================ Human Message ===========================


The escalation criteria is: There's an immediate risk of electrical, water,
or fire damage

Here's the email:


    Date: January 10, 2025
    From: City of Los Angeles Building and Safety Department
    To: West Coast Development, project 345678123 - Sunset Luxury Condominiums
    Location: Los Angeles, CA

    Following an inspection of your site at 456 Sunset Boulevard, we have
    identified
    the following building code
    violations:

    Electrical Wiring: Exposed wiring was found in the underground parking
    garage, posing
    a safety hazard.
    Fire Safety: Insufficient fire extinguishers were available across
    multiple floors of
    the structure under
    construction.
    Structural Integrity: The temporary support beams in the eastern wing
    do not meet the
    load-bearing standards
    specified in local building codes.
    Required Corrective Actions:

    Replace or properly secure exposed wiring to meet electrical safety
    standards.
    Install additional fire extinguishers in compliance with fire code
    requirements.
...
    Contact: For questions or to schedule a re-inspection, please contact
    the Building and
    Safety Department at
    (555) 456-7890 or email inspections@lacity.gov.


2025-01-26 13:19:14,234 - INFO - Calling the email notice
extraction graph...
2025-01-26 13:19:14,237 - INFO - Parsing notice...
================================== Ai Message ==========================
Tool Calls:
  extract_notice_data (call_xd8mvZQmuhUOsoOden6Mrl1z)
 Call ID: call_xd8mvZQmuhUOsoOden6Mrl1z
  Args:
    email:
    Date: January 10, 2025
    From: City of Los Angeles Building and Safety Department
    To: West Coast Development, project 345678123 - Sunset Luxury
    Condominiums
    Location: Los Angeles, CA

    Following an inspection of your site at 456 Sunset Boulevard, we have
    identified the following building code
    violations:

    Electrical Wiring: Exposed wiring was found in the underground parking
    garage, posing a safety hazard.
    Fire Safety: Insufficient fire extinguishers were available across
    multiple floors of the structure under
    construction.
    Structural Integrity: The temporary support beams in the eastern wing
    do not meet the load-bearing standards
    specified in local building codes.
    Required Corrective Actions:

    Replace or properly secure exposed wiring to meet electrical safety
    standards.
    Install additional fire extinguishers in compliance with fire code
    requirements.
    Reinforce or replace temporary support beams to ensure structural
    stability.
    Deadline for Compliance: Violations must be addressed no later than
    February 5, 2025. Failure to comply may result in
...

    Contact: For questions or to schedule a re-inspection, please
    contact the Building and Safety Department at
    (555) 456-7890 or email inspections@lacity.gov.
    escalation_criteria: There's an immediate risk of electrical,
    water, or fire damage
2025-01-26 13:19:16,368 - INFO - Determining escalation status...
2025-01-26 13:19:16,982 - INFO - Escalation needed!
2025-01-26 13:19:16,987 - INFO - Sending escalation emails...
2025-01-26 13:19:17,988 - INFO - Escalation email sent to
brog@abc.com
2025-01-26 13:19:18,995 - INFO - Escalation email sent to
bigceo@company.com
2025-01-26 13:19:18,998 - INFO - Creating legal ticket for notice...
2025-01-26 13:19:21,004 - INFO - Follow-up is required before
creating this ticket
2025-01-26 13:19:21,682 - INFO - Creating legal ticket for notice...
2025-01-26 13:19:23,686 - INFO - Legal ticket created!
================================= Tool Message ==========================
Name: extract_notice_data

date_of_notice=datetime.date(2025, 1, 10) entity_name='City of Los Angeles
Building and Safety Department'
entity_phone='(555) 456-7890' entity_email='inspections@lacity.gov'
project_id=345678123
site_location='Los Angeles, CA' violation_type='Electrical Wiring,
Fire Safety, Structural Integrity'
equired_changes='Replace or properly secure exposed wiring to meet
electrical safety standards.
Install additional fire extinguishers in compliance with fire code
requirements. Reinforce or
replace temporary support beams to ensure structural stability.'
compliance_deadline=datetime.date(2025, 2, 5) max_potential_fine=None
================================== Ai Message =======================

The email from the City of Los Angeles Building and Safety Department
contains serious
violations that pose an immediate risk of electrical and fire damage.
Here are the details
extracted from the notice:

- **Date of Notice:** January 10, 2025
- **Entity Name:** City of Los Angeles Building and Safety Department
- **Entity Phone:** (555) 456-7890
- **Entity Email:** inspections@lacity.gov
- **Project ID:** 345678123
- **Site Location:** Los Angeles, CA
- **Violation Types:** Electrical Wiring, Fire Safety, Structural
Integrity
- **Required Changes:**
  - Replace or properly secure exposed wiring to meet electrical
  safety standards.
  - Install additional fire extinguishers in compliance with fire
  code requirements.
  - Reinforce or replace temporary support beams to ensure structural
  stability.
- **Compliance Deadline:** February 5, 2025

Given the nature of the violations, it is crucial to address these
issues immediately to avoid
potential stop-work orders and fines. Would you like to take any
specific actions regarding
this notice?